# 第7章: アンサンブル学習で複数モデルを組み合わせる

この Notebook は、原本 `machine-learning-book/ch07/ch07.ipynb` を最新の Python / scikit-learn / xgboost 環境で継続検証しやすい形に移行したものです。  
多数決、Bagging、AdaBoost、Gradient Boosting、XGBoost という章の主題を、CI でヘッドレス実行できる Notebook として再構成しています。


## この Notebook で確認すること

- 現在の `uv` 環境で主要パッケージのバージョンを確認する。
- 読み取り専用サブモジュール `machine-learning-book/ch07/` から図版と `wine.data` を参照できることを確認する。
- アンサンブル誤差の理論式と、多数決分類器の実装・評価・チューニングを再現する。
- Wine データセットで Bagging、AdaBoost、Gradient Boosting、XGBoost を比較する。
- `pytest --nbmake` 前提で停止しない描画と軽量な学習設定に調整する。


In [ ]:
from importlib.metadata import version
from itertools import product
from math import ceil, comb, log
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn import datasets
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
)
from sklearn.metrics import accuracy_score, auc, roc_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


In [ ]:
PACKAGE_NAMES = [
    "ipykernel",
    "matplotlib",
    "nbmake",
    "numpy",
    "pandas",
    "pytest",
    "scikit-learn",
    "xgboost",
]


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "machine-learning-book").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("リポジトリルートを特定できませんでした。")


repo_root = find_repo_root(Path.cwd())
chapter_root = repo_root / "machine-learning-book" / "ch07"
figure_dir = chapter_root / "figures"
wine_path = chapter_root / "wine.data"

if not wine_path.exists():
    raise FileNotFoundError(f"wine.data が見つかりません: {wine_path}")

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Matplotlib backend: {matplotlib.get_backend()}")

try:
    import xgboost as xgb

    XGBOOST_AVAILABLE = True
    xgboost_status = xgb.__version__
except Exception as exc:
    xgb = None
    XGBOOST_AVAILABLE = False
    xgboost_status = f"unavailable ({type(exc).__name__}: {exc})"

print(f"xgboost: {xgboost_status}")


In [ ]:
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=["パッケージ", "バージョン"],
)
package_versions


## 原本図版とデータの参照

移行版 Notebook の実装コードは `src/ch07/` 側に置き、書籍の図版と Wine データは読み取り専用サブモジュールから参照します。  
ここでは代表的な図版を表示し、以降の節で使う `wine.data` の存在も確認します。


In [ ]:
selected_figures = [
    ("07_03.png", 420),
    ("07_08.png", 420),
    ("07_11.png", 420),
]

for name, width in selected_figures:
    path = figure_dir / name
    if not path.exists():
        raise FileNotFoundError(f"図版が見つかりません: {path}")
    display(Image(filename=str(path), width=width))

pd.Series({"wine.data": wine_path.exists(), "rows": sum(1 for _ in wine_path.open())})


## アンサンブル誤差の理論確認

原本と同様に、多数決アンサンブルで過半数が誤る確率を二項分布で計算します。  
単体分類器の誤差率が 0.5 未満であれば、アンサンブル化によって誤差が下がることを可視化します。


In [ ]:
def ensemble_error(n_classifiers: int, error: float) -> float:
    start = int(ceil(n_classifiers / 2))
    return sum(
        comb(n_classifiers, k) * (error**k) * ((1 - error) ** (n_classifiers - k))
        for k in range(start, n_classifiers + 1)
    )


error_range = np.linspace(0.0, 1.0, 101)
ensemble_errors = np.array([ensemble_error(n_classifiers=11, error=err) for err in error_range])

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(error_range, ensemble_errors, linewidth=2, label="Ensemble error")
ax.plot(error_range, error_range, linestyle="--", linewidth=2, label="Base error")
ax.set_xlabel("Base error")
ax.set_ylabel("Error")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

pd.Series(
    {
        "base_error=0.25 の ensemble_error": round(ensemble_error(11, 0.25), 4),
        "base_error=0.49 の ensemble_error": round(ensemble_error(11, 0.49), 4),
    }
)


## 多数決分類器の実装

この節では原本の教育意図を残すため、`VotingClassifier` を使うだけでなく、シンプルな `MajorityVoteClassifier` を自前で実装します。  
`GridSearchCV` で扱えるよう、パラメータ公開も含めて scikit-learn 互換の Estimator として定義します。


In [ ]:
class MajorityVoteClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, classifiers, vote="classlabel", weights=None):
        self.classifiers = classifiers
        self.vote = vote
        self.weights = weights

    def fit(self, X, y):
        if self.vote not in {"classlabel", "probability"}:
            raise ValueError("vote must be 'classlabel' or 'probability'.")
        if self.weights is not None and len(self.weights) != len(self.classifiers):
            raise ValueError("weights の長さは classifiers と一致する必要があります。")

        self.label_encoder_ = LabelEncoder()
        encoded_y = self.label_encoder_.fit_transform(y)
        self.classes_ = self.label_encoder_.classes_

        self.named_classifiers_ = {}
        self.classifiers_ = []
        used_names = {}
        for clf in self.classifiers:
            base_name = clf.__class__.__name__.lower()
            count = used_names.get(base_name, 0)
            used_names[base_name] = count + 1
            name = base_name if count == 0 else f"{base_name}-{count + 1}"
            fitted = clone(clf).fit(X, encoded_y)
            self.named_classifiers_[name] = fitted
            self.classifiers_.append(fitted)
        return self

    def predict(self, X):
        if self.vote == "probability":
            maj_vote = np.argmax(self.predict_proba(X), axis=1)
        else:
            predictions = np.asarray([clf.predict(X) for clf in self.classifiers_]).T
            maj_vote = np.apply_along_axis(
                lambda x: np.argmax(np.bincount(x, weights=self.weights)),
                axis=1,
                arr=predictions,
            )
        return self.label_encoder_.inverse_transform(maj_vote)

    def predict_proba(self, X):
        probas = np.asarray([clf.predict_proba(X) for clf in self.classifiers_])
        return np.average(probas, axis=0, weights=self.weights)

    def get_params(self, deep=True):
        if not deep:
            return {"classifiers": self.classifiers, "vote": self.vote, "weights": self.weights}
        params = super().get_params(deep=False)
        used_names = {}
        for clf in self.classifiers:
            clf_name = clf.__class__.__name__.lower()
            count = used_names.get(clf_name, 0)
            used_names[clf_name] = count + 1
            key = clf_name if count == 0 else f"{clf_name}-{count + 1}"
            params[key] = clf
            for sub_key, value in clf.get_params(deep=True).items():
                params[f"{key}__{sub_key}"] = value
        return params


## 多数決分類器の評価とチューニング

原本と同じく Iris の 2 クラス部分集合を使い、ロジスティック回帰、決定木、KNN と自作多数決分類器を比較します。  
交差検証の fold 数とグリッドサイズは、`nbmake` で現実的な時間に収まるよう縮小しています。


In [ ]:
iris = datasets.load_iris()
X_iris = iris.data[50:, [1, 2]]
y_iris = iris.target[50:]
y_iris = LabelEncoder().fit_transform(y_iris)

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris,
    y_iris,
    test_size=0.5,
    random_state=1,
    stratify=y_iris,
)

clf1 = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(C=0.001, solver="lbfgs", random_state=1)),
    ]
)
clf2 = DecisionTreeClassifier(max_depth=1, criterion="entropy", random_state=0)
clf3 = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=1)),
    ]
)
mv_clf = MajorityVoteClassifier(classifiers=[clf1, clf2, clf3])

clf_labels = ["Logistic regression", "Decision tree", "KNN", "Majority voting"]
all_clf = [clf1, clf2, clf3, mv_clf]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

roc_summary = []
for label, clf in zip(clf_labels, all_clf):
    scores = cross_val_score(clf, X_train_iris, y_train_iris, cv=cv, scoring="roc_auc", n_jobs=1)
    roc_summary.append({"classifier": label, "roc_auc_mean": scores.mean(), "roc_auc_std": scores.std()})

display(pd.DataFrame(roc_summary))

fig, ax = plt.subplots(figsize=(5.8, 4.0))
styles = [
    ("black", ":"),
    ("orange", "--"),
    ("blue", "-."),
    ("green", "-"),
]
for clf, label, (color, linestyle) in zip(all_clf, clf_labels, styles):
    y_score = clf.fit(X_train_iris, y_train_iris).predict_proba(X_test_iris)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_iris, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linestyle=linestyle, label=f"{label} (AUC={roc_auc:.2f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_iris)

region_models = [
    Pipeline([("identity", "passthrough"), ("lr", LogisticRegression(C=0.001, solver="lbfgs", random_state=1))]),
    DecisionTreeClassifier(max_depth=1, criterion="entropy", random_state=0),
    Pipeline([("identity", "passthrough"), ("knn", KNeighborsClassifier(n_neighbors=1))]),
    MajorityVoteClassifier(
        classifiers=[
            Pipeline([("identity", "passthrough"), ("lr", LogisticRegression(C=0.001, solver="lbfgs", random_state=1))]),
            DecisionTreeClassifier(max_depth=1, criterion="entropy", random_state=0),
            Pipeline([("identity", "passthrough"), ("knn", KNeighborsClassifier(n_neighbors=1))]),
        ]
    ),
]

x_min, x_max = X_train_std[:, 0].min() - 1, X_train_std[:, 0].max() + 1
y_min, y_max = X_train_std[:, 1].min() - 1, X_train_std[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))

fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.2), sharex=True, sharey=True)
for ax, clf, title in zip(axes.ravel(), region_models, clf_labels):
    clf.fit(X_train_std, y_train_iris)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25)
    ax.scatter(X_train_std[y_train_iris == 0, 0], X_train_std[y_train_iris == 0, 1], c="blue", marker="^", s=35)
    ax.scatter(X_train_std[y_train_iris == 1, 0], X_train_std[y_train_iris == 1, 1], c="green", marker="o", s=35)
    ax.set_title(title)

fig.supxlabel("Sepal width [standardized]")
fig.supylabel("Petal length [standardized]")
plt.tight_layout()
plt.show()


In [ ]:
param_grid = {
    "decisiontreeclassifier__max_depth": [1, 2],
    "pipeline__lr__C": [0.001, 0.1, 10.0],
}

grid = GridSearchCV(
    estimator=mv_clf,
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=1,
)
grid.fit(X_train_iris, y_train_iris)

grid_summary = pd.DataFrame(
    {
        "mean_test_score": grid.cv_results_["mean_test_score"],
        "std_test_score": grid.cv_results_["std_test_score"],
        "params": [str(params) for params in grid.cv_results_["params"]],
    }
)

display(grid_summary)
pd.Series(
    {
        "best_params": str(grid.best_params_),
        "best_score": round(grid.best_score_, 4),
    }
)


## Wine データセットでの Bagging と Boosting

原本は Wine データセットの 2 クラス問題に対して Bagging と AdaBoost を適用していました。  
移行版でも `wine.data` をローカルファイルから読み込み、同じ 2 特徴量で境界と精度を比較します。


In [ ]:
wine_columns = [
    "Class label",
    "Alcohol",
    "Malic acid",
    "Ash",
    "Alcalinity of ash",
    "Magnesium",
    "Total phenols",
    "Flavanoids",
    "Nonflavanoid phenols",
    "Proanthocyanins",
    "Color intensity",
    "Hue",
    "OD280/OD315 of diluted wines",
    "Proline",
]

df_wine = pd.read_csv(wine_path, header=None, names=wine_columns)
df_wine = df_wine[df_wine["Class label"] != 1].copy()

X_wine = df_wine[["Alcohol", "OD280/OD315 of diluted wines"]].to_numpy()
y_wine = LabelEncoder().fit_transform(df_wine["Class label"].to_numpy())

X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine,
    y_wine,
    test_size=0.2,
    random_state=1,
    stratify=y_wine,
)

pd.Series(
    {
        "samples": len(df_wine),
        "features": X_wine.shape[1],
        "train_size": len(X_train_wine),
        "test_size": len(X_test_wine),
    }
)


In [ ]:
tree = DecisionTreeClassifier(criterion="entropy", max_depth=None, random_state=1)
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(criterion="entropy", max_depth=None, random_state=1),
    n_estimators=100,
    max_samples=1.0,
    max_features=1.0,
    bootstrap=True,
    bootstrap_features=False,
    n_jobs=1,
    random_state=1,
)
stump = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=1)
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=1),
    n_estimators=100,
    learning_rate=0.1,
    random_state=1,
)

models = {
    "Decision tree": tree,
    "Bagging": bag,
    "Decision stump": stump,
    "AdaBoost": ada,
}

rows = []
for name, model in models.items():
    fitted = clone(model).fit(X_train_wine, y_train_wine)
    rows.append(
        {
            "model": name,
            "train_accuracy": accuracy_score(y_train_wine, fitted.predict(X_train_wine)),
            "test_accuracy": accuracy_score(y_test_wine, fitted.predict(X_test_wine)),
        }
    )

display(pd.DataFrame(rows))

x_min, x_max = X_train_wine[:, 0].min() - 1, X_train_wine[:, 0].max() + 1
y_min, y_max = X_train_wine[:, 1].min() - 1, X_train_wine[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))

fig, axes = plt.subplots(2, 2, figsize=(8.0, 6.0), sharex=True, sharey=True)
for ax, (name, model) in zip(axes.ravel(), models.items()):
    fitted = clone(model).fit(X_train_wine, y_train_wine)
    Z = fitted.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25)
    ax.scatter(X_train_wine[y_train_wine == 0, 0], X_train_wine[y_train_wine == 0, 1], c="blue", marker="^", s=35)
    ax.scatter(X_train_wine[y_train_wine == 1, 0], X_train_wine[y_train_wine == 1, 1], c="green", marker="o", s=35)
    ax.set_title(name)

fig.supxlabel("Alcohol")
fig.supylabel("OD280/OD315 of diluted wines")
plt.tight_layout()
plt.show()


## AdaBoost の重み更新を小さな例で確認する

原本にある 10 サンプルの例をそのまま再現し、誤分類されたサンプルの重みがどのように強調されるかを数式どおりに確認します。


In [ ]:
y = np.array([1, 1, 1, -1, -1, -1, 1, 1, 1, -1])
y_hat = np.array([1, 1, 1, -1, -1, -1, -1, -1, -1, -1])
correct = y == y_hat
weights = np.full(10, 0.1)
epsilon = np.mean(~correct)
alpha = 0.5 * log((1 - epsilon) / epsilon)
updated_weights = np.where(correct, weights * np.exp(-alpha), weights * np.exp(alpha))
normalized_weights = updated_weights / updated_weights.sum()

pd.DataFrame(
    {
        "y": y,
        "y_hat": y_hat,
        "correct": correct,
        "initial_weight": weights,
        "updated_weight": updated_weights.round(4),
        "normalized_weight": normalized_weights.round(4),
    }
)


## Gradient Boosting と XGBoost

原本後半の焦点は勾配ブースティング系手法の比較です。  
移行版では `GradientBoostingClassifier`、`VotingClassifier` による単純平均、そして `XGBClassifier` を同じ Wine データ分割で比較します。`xgboost` のネイティブ実行基盤が使えない環境では、Notebook を止めずに `HistGradientBoostingClassifier` へ自動フォールバックします。


In [ ]:
soft_voting = VotingClassifier(
    estimators=[
        ("lr", Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=1000, random_state=1))])),
        ("tree", DecisionTreeClassifier(max_depth=3, random_state=1)),
        ("knn", Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))])),
    ],
    voting="soft",
)

gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=1,
)

xgb_clf = xgb.XGBClassifier(
    n_estimators=80,
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=1,
    n_jobs=1,
) if XGBOOST_AVAILABLE else HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=3,
    max_iter=120,
    random_state=1,
)

comparison_models = {
    "Soft voting": soft_voting,
    "GradientBoosting": gb_clf,
    "XGBoost" if XGBOOST_AVAILABLE else "HistGradientBoosting (fallback)": xgb_clf,
}

comparison_rows = []
for name, model in comparison_models.items():
    fitted = clone(model).fit(X_train_wine, y_train_wine)
    comparison_rows.append(
        {
            "model": name,
            "train_accuracy": accuracy_score(y_train_wine, fitted.predict(X_train_wine)),
            "test_accuracy": accuracy_score(y_test_wine, fitted.predict(X_test_wine)),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

fig, ax = plt.subplots(figsize=(6.0, 3.6))
bars = ax.bar(comparison_df["model"], comparison_df["test_accuracy"], color=["#4C72B0", "#55A868", "#C44E52"])
ax.bar_label(bars, fmt="%.3f")
ax.set_ylim(0.7, 1.02)
ax.set_ylabel("Test accuracy")
ax.set_title("Gradient boosting family comparison")
plt.tight_layout()
plt.show()


## まとめ

この移行版では、原本 `ch07` の教育的意図を保ちながら、次の点を最新環境向けに更新しました。

- 外部 URL への依存を避け、`wine.data` と図版をサブモジュールから読み取り専用で参照する構成にした。
- `BaggingClassifier` と `AdaBoostClassifier` は現行 scikit-learn の `estimator=` API に合わせた。
- 多数決分類器は自前実装を残しつつ、`GridSearchCV` に掛けられる形へ整理した。
- `xgboost` を依存関係に追加し、勾配ブースティング節も Notebook 単体で再現できるようにした。
- 推定器数や交差検証の fold 数は、`pytest --nbmake` で継続検証しやすいサイズへ調整した。
